# Data Augmentation, BatchNorm, and Transfer Learning

## One simple applied project

A warehouse camera reads a handwritten parcel digit.

- **Source task:** identify the digit from `0` to `9`.
- **Target task:** send the parcel to route A, B, or C.

```text
many source labels → learn useful digit features
few target labels  → reuse those features for routing
```

This notebook uses the small Digits dataset and runs without downloading data or pretrained weights.

## 1. Setup

Import the required libraries, fix random seeds, and choose an available device.

In [ ]:
# WHAT: Import libraries, fix seeds, and choose a device.
# WHY: Reproducible experiments need a known random state and environment.
# OUTPUT: Library versions and the selected device.

from copy import deepcopy
from pathlib import Path
import random
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)
print("device:", DEVICE)

## 2. Load and split the data

The images are `8 × 8` grayscale digits.

We first separate original examples into source and target pools. This prevents the source model from seeing target validation or test images.

Target routes:

```text
Route A = digits 0–3
Route B = digits 4–6
Route C = digits 7–9
```

In [ ]:
# WHAT: Load images and create disjoint source and target pools.
# WHY: Transfer evaluation must use target examples unseen during source pretraining.
# OUTPUT: Image shape and source/target pool sizes.

digits = load_digits()
images = (digits.images.astype("float32") / 16.0)[:, None, :, :]
digit_labels = digits.target.astype("int64")
all_indices = np.arange(len(images))

source_indices, target_indices = train_test_split(
    all_indices,
    test_size=0.35,
    stratify=digit_labels,
    random_state=SEED,
)

source_train_idx, source_valid_idx = train_test_split(
    source_indices,
    test_size=0.18,
    stratify=digit_labels[source_indices],
    random_state=SEED,
)

print("image shape:", images.shape)
print("pixel range:", float(images.min()), "to", float(images.max()))
print("source train/valid:", len(source_train_idx), len(source_valid_idx))
print("target pool:", len(target_indices))
print("source-target overlap:", len(set(source_indices) & set(target_indices)))

In [ ]:
# WHAT: Create the three route labels and target train/validation/test splits.
# WHY: A small target-train split makes transfer learning useful to compare.
# OUTPUT: Target split sizes and class counts.

def route_group(digits_array):
    digits_array = np.asarray(digits_array)
    return np.where(digits_array <= 3, 0, np.where(digits_array <= 6, 1, 2)).astype("int64")

target_train_idx, target_rest_idx = train_test_split(
    target_indices,
    train_size=150,
    stratify=route_group(digit_labels[target_indices]),
    random_state=SEED,
)
target_valid_idx, target_test_idx = train_test_split(
    target_rest_idx,
    train_size=160,
    stratify=route_group(digit_labels[target_rest_idx]),
    random_state=SEED,
)

ROUTE_NAMES = ["A: digits 0-3", "B: digits 4-6", "C: digits 7-9"]
for name, indices in [
    ("target train", target_train_idx),
    ("target valid", target_valid_idx),
    ("target test", target_test_idx),
]:
    counts = np.bincount(route_group(digit_labels[indices]), minlength=3)
    print(f"{name:12s}: {len(indices):3d} examples, counts={counts.tolist()}")

In [ ]:
# WHAT: Display one example for each digit.
# WHY: Image inspection should happen before preprocessing or model building.
# OUTPUT: Ten labeled grayscale images.

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for digit, axis in enumerate(axes.flat):
    index = np.flatnonzero(digit_labels == digit)[0]
    axis.imshow(images[index, 0], cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"digit {digit}")
    axis.axis("off")
plt.tight_layout()

## 3. Common augmentations

| Transform | Common use | Important warning |
|---|---|---|
| Horizontal flip | animals and natural objects | unsafe for text and digits |
| Small rotation | camera or scanner tilt | avoid when direction is the label |
| Random crop | different object position/size | do not remove the important object |
| Color jitter | changing light | avoid when exact color matters |
| Blur | focus variation | may remove tiny details |
| Random erasing | partial occlusion | may erase the defect |
| Noise | camera/sensor noise | keep it mild |

For digits, this notebook uses built-in `RandomAffine` for a small translation and `ColorJitter` for mild contrast change.

We deliberately do **not** flip or rotate digits: a flip is not a normal handwritten digit, and a large rotation can make `6` look like `9`.

```text
training   → random built-in augmentation
validation → no random augmentation
test       → no random augmentation
```

In [ ]:
# WHAT: Create a built-in, label-safe augmentation pipeline for digit images.
# WHY: A parcel digit can be slightly off-centre or captured with different contrast.
# OUTPUT: The same number of images with the same [batch, channel, height, width] shape.

# v2.Compose means: apply the first transform, then pass its result to the next transform.
digit_augmentation = v2.Compose([
    # RandomAffine can shift, rotate, zoom, or shear an image.
    v2.RandomAffine(
        degrees=0,                 # keep angle at 0: do not rotate digits such as 6 and 9
        translate=(1 / 8, 1 / 8),  # allow up to 12.5% shift = one pixel in an 8 x 8 image
        fill=0,                    # fill newly empty pixels with 0 = black after the shift
    ),

    # ColorJitter changes image appearance, not the digit shape.
    # contrast=0.1 randomly chooses a contrast factor from 0.9 to 1.1.
    v2.ColorJitter(contrast=0.1),
])

def augment_batch(batch):
    # batch has shape [batch_size, 1, 8, 8].
    # Give every image its own random shift and contrast instead of changing all images equally.
    changed_images = [digit_augmentation(image) for image in batch]

    # Stack individual [1, 8, 8] images back into one [batch_size, 1, 8, 8] tensor.
    return torch.stack(changed_images)

# Use eight target-training images only. Never augment validation or test images.
example_batch = torch.tensor(images[target_train_idx[:8]])

# Create changed copies. The original images inside `images` are not modified.
augmented = augment_batch(example_batch)

# Check the two most important data contracts after augmentation.
print("shape:", augmented.shape)  # should stay [8, 1, 8, 8]
print("range:", float(augmented.min()), "to", float(augmented.max()))  # should stay in [0, 1]

In [ ]:
# WHAT: Show repeated training views beside a fixed evaluation view.
# WHY: Augmentation must be visible, mild, and limited to training.
# OUTPUT: Random top-row views and identical bottom-row views.

base_image = torch.tensor(images[target_train_idx[0]])
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for column in range(6):
    train_view = augment_batch(base_image.unsqueeze(0))[0]
    axes[0, column].imshow(train_view[0], cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title("train view")
    axes[1, column].imshow(base_image[0], cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title("eval view")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()

## 4. Input normalization with a simple example

Pixel scaling has already changed the raw range to `[0,1]`.

Standardization uses:

\[
x' = \frac{x - mean}{std}
\]

Example:

```text
values = [0.0, 0.5, 1.0]
mean = 0.5, std = 0.5
result = [-1, 0, 1]
```

We calculate mean and standard deviation from source training images only. The transferred backbone expects the same values later.

In [ ]:
# WHAT: Calculate source-training input statistics and show a numeric example.
# WHY: Training and inference must use the same input scale.
# OUTPUT: Source mean/std and normalized values [-1,0,1] for the small example.

SOURCE_MEAN = float(images[source_train_idx].mean())
SOURCE_STD = float(images[source_train_idx].std())

def normalize_batch(batch):
    return (batch - SOURCE_MEAN) / SOURCE_STD

simple_values = torch.tensor([0.0, 0.5, 1.0])
simple_result = (simple_values - 0.5) / 0.5

print(f"source image mean={SOURCE_MEAN:.4f}, std={SOURCE_STD:.4f}")
print("simple input:", simple_values.tolist())
print("simple normalized:", simple_result.tolist())

## 5. Batch Normalization with a small example

BatchNorm works inside the network.

```text
hidden values [10, 12, 14]
        ↓ BatchNorm
values roughly centered around 0
```

Then the layer learns:

- `gamma`: scale;
- `beta`: shift.

Most important rule:

```python
model.train()  # uses current batch and updates running statistics
model.eval()   # uses remembered running statistics
```

In [ ]:
# WHAT: Pass simple values through BatchNorm in training and evaluation modes.
# WHY: A numeric example makes centering and train/eval behavior visible.
# OUTPUT: Near-zero training output mean, running mean, and eval-mode output.

batch_norm = nn.BatchNorm1d(1)
hidden_values = torch.tensor([[10.0], [12.0], [14.0]])

batch_norm.train()
train_output = batch_norm(hidden_values)
print("input values:", hidden_values.squeeze().tolist())
print("training output:", train_output.detach().squeeze().round(decimals=3).tolist())
print("training output mean:", round(float(train_output.mean()), 6))
print("remembered running mean:", round(float(batch_norm.running_mean.item()), 3))

batch_norm.eval()
with torch.inference_mode():
    eval_output = batch_norm(hidden_values)
print("evaluation output:", eval_output.squeeze().round(decimals=3).tolist())

## 6. Create datasets and loaders

The dataset returns raw `[0,1]` tensors. Training may augment them, then every split uses the same input normalization.

In [ ]:
# WHAT: Build reusable datasets and deterministic data loaders.
# WHY: Every model comparison must use the same splits and label mapping.
# OUTPUT: Verified source and target batch shapes.

class ArrayDataset(Dataset):
    def __init__(self, image_array, labels):
        self.images = torch.tensor(image_array, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.images[index], self.labels[index]

def make_loader(dataset, shuffle=False, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=32,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )

source_train_loader = make_loader(
    ArrayDataset(images[source_train_idx], digit_labels[source_train_idx]),
    shuffle=True,
    seed=SEED + 1,
)
source_valid_loader = make_loader(
    ArrayDataset(images[source_valid_idx], digit_labels[source_valid_idx])
)
target_train_loader = make_loader(
    ArrayDataset(images[target_train_idx], route_group(digit_labels[target_train_idx])),
    shuffle=True,
    seed=SEED + 2,
)
target_valid_loader = make_loader(
    ArrayDataset(images[target_valid_idx], route_group(digit_labels[target_valid_idx]))
)
target_test_loader = make_loader(
    ArrayDataset(images[target_test_idx], route_group(digit_labels[target_test_idx]))
)

source_batch = next(iter(source_train_loader))
target_batch = next(iter(target_train_loader))
print("source batch:", source_batch[0].shape, source_batch[1].shape)
print("target batch:", target_batch[0].shape, target_batch[1].shape)

## 7. Backbone and head

```text
image → block 1 → block 2 → features → head → logits
```

- **Backbone:** block 1 and block 2; learns image features.
- **Head:** final linear layer; produces task classes.

The source head has 10 outputs. The target head has 3 outputs.

In [ ]:
# WHAT: Define a small CNN with two backbone blocks and one replaceable head.
# WHY: Clear separation makes freezing and fine-tuning easy to understand.
# OUTPUT: A model that can support either 10 source classes or 3 target classes.

class SmallTransferCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Linear(32 * 2 * 2, num_classes)

    def forward_features(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return x.flatten(1)

    def forward(self, x):
        return self.head(self.forward_features(x))

def parameter_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

demo_model = SmallTransferCNN(num_classes=10).to(DEVICE)
demo_input = normalize_batch(torch.zeros(4, 1, 8, 8, device=DEVICE))
print("logit shape:", demo_model(demo_input).shape)
print("total/trainable parameters:", parameter_counts(demo_model))

## 8. Training and evaluation functions

Training order:

```text
augment → normalize → forward → loss → backward → optimizer step
```

Validation uses `model.eval()` and does not apply random augmentation.

In [ ]:
# WHAT: Define one training epoch and one deterministic evaluation pass.
# WHY: All strategies should use the same loss and metric calculations.
# OUTPUT: Reusable functions returning loss, accuracy, labels, and predictions.

def train_one_epoch(model, loader, optimizer, *, use_augmentation, frozen_modules=()):
    model.train()
    for module in frozen_modules:
        module.eval()

    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, count = 0.0, 0, 0
    for raw_images, labels in loader:
        if use_augmentation:
            raw_images = augment_batch(raw_images)
        inputs = normalize_batch(raw_images).to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        count += labels.size(0)
    return loss_sum / count, correct / count

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, count = 0.0, 0
    labels_all, predictions_all = [], []
    for raw_images, labels in loader:
        inputs = normalize_batch(raw_images).to(DEVICE)
        labels_device = labels.to(DEVICE)
        logits = model(inputs)
        loss = criterion(logits, labels_device)

        loss_sum += loss.item() * labels.size(0)
        count += labels.size(0)
        labels_all.append(labels)
        predictions_all.append(logits.argmax(1).cpu())

    labels_array = torch.cat(labels_all).numpy()
    predictions_array = torch.cat(predictions_all).numpy()
    return {
        "loss": loss_sum / count,
        "accuracy": float((labels_array == predictions_array).mean()),
        "labels": labels_array,
        "predictions": predictions_array,
    }

In [ ]:
# WHAT: Train a model, keep its best validation state, and stop when improvement stalls.
# WHY: The last epoch is not automatically the best model.
# OUTPUT: Restored best weights, learning history, and summary metrics.

def fit(
    model,
    train_loader,
    valid_loader,
    optimizer,
    *,
    epochs,
    use_augmentation,
    frozen_modules=(),
    patience=6,
):
    history = {"train_loss": [], "valid_loss": [], "valid_accuracy": []}
    best_state = deepcopy(model.state_dict())
    best_loss = float("inf")
    best_epoch = 0
    stale = 0
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, _ = train_one_epoch(
            model,
            train_loader,
            optimizer,
            use_augmentation=use_augmentation,
            frozen_modules=frozen_modules,
        )
        valid = evaluate(model, valid_loader)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid["loss"])
        history["valid_accuracy"].append(valid["accuracy"])

        if valid["loss"] < best_loss - 1e-4:
            best_loss = valid["loss"]
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    best_valid = evaluate(model, valid_loader)
    return {
        "history": history,
        "best_epoch": best_epoch,
        "valid_loss": best_valid["loss"],
        "valid_accuracy": best_valid["accuracy"],
        "seconds": time.perf_counter() - start,
    }

## 9. Transfer learning in one picture

```text
SOURCE MODEL
image → learned backbone → 10-class digit head

TARGET MODEL
image → same backbone    → new 3-class route head
```

### Three choices

1. **Scratch:** random backbone and random head.
2. **Feature extraction:** freeze backbone; train new head.
3. **Partial fine-tuning:** unfreeze the last block with a smaller learning rate.

## 10. Source pretraining

The source task teaches the backbone to recognize strokes and digit shapes. The best source validation state becomes our pretrained model.

In [ ]:
# WHAT: Pretrain the 10-class source model.
# WHY: The learned backbone will provide reusable digit features.
# OUTPUT: Best source validation accuracy and epoch.

torch.manual_seed(SEED)
source_model = SmallTransferCNN(num_classes=10).to(DEVICE)
source_optimizer = torch.optim.AdamW(source_model.parameters(), lr=3e-3, weight_decay=1e-4)
source_result = fit(
    source_model,
    source_train_loader,
    source_valid_loader,
    source_optimizer,
    epochs=16,
    use_augmentation=True,
)
print(
    f"source best epoch={source_result['best_epoch']}, "
    f"validation accuracy={source_result['valid_accuracy']:.3f}"
)

## 11. Target baseline: train from scratch

This model uses only the small target training set. It tells us what the target data can learn without transferred features.

In [ ]:
# WHAT: Train a three-class target model from random initialization.
# WHY: A scratch baseline is needed to measure the value of transfer learning.
# OUTPUT: Scratch validation loss, accuracy, and trainable parameter count.

torch.manual_seed(SEED + 10)
scratch_model = SmallTransferCNN(num_classes=3).to(DEVICE)
scratch_optimizer = torch.optim.AdamW(scratch_model.parameters(), lr=3e-3, weight_decay=1e-4)
scratch_result = fit(
    scratch_model,
    target_train_loader,
    target_valid_loader,
    scratch_optimizer,
    epochs=22,
    use_augmentation=True,
)
print("scratch:", scratch_result | {"history": "omitted"})
print("total/trainable:", parameter_counts(scratch_model))

## 12. Transfer strategy 1: freeze the backbone

Steps:

1. copy the source model;
2. replace the 10-class head with a new 3-class head;
3. freeze block 1 and block 2;
4. train only the new head.

The frozen blocks stay in evaluation mode so their BatchNorm running statistics remain fixed.

In [ ]:
# WHAT: Create a target model with a frozen source backbone and new head.
# WHY: Feature extraction tests whether source features work without changing them.
# OUTPUT: Only the new three-class head appears as trainable.

frozen_model = deepcopy(source_model)
frozen_model.head = nn.Linear(frozen_model.head.in_features, 3)
for parameter in frozen_model.block1.parameters():
    parameter.requires_grad = False
for parameter in frozen_model.block2.parameters():
    parameter.requires_grad = False
frozen_model = frozen_model.to(DEVICE)

print("total/trainable:", parameter_counts(frozen_model))
print("trainable tensors:", [name for name, p in frozen_model.named_parameters() if p.requires_grad])

In [ ]:
# WHAT: Train only the new target head.
# WHY: This is the fastest and simplest transfer-learning baseline.
# OUTPUT: Frozen-backbone validation loss and accuracy.

frozen_optimizer = torch.optim.AdamW(frozen_model.head.parameters(), lr=3e-3)
frozen_result = fit(
    frozen_model,
    target_train_loader,
    target_valid_loader,
    frozen_optimizer,
    epochs=22,
    use_augmentation=True,
    frozen_modules=(frozen_model.block1, frozen_model.block2),
)
print("frozen transfer:", frozen_result | {"history": "omitted"})

## 13. Transfer strategy 2: fine-tune the last block

Start from the best frozen model.

```text
block 1 → frozen
block 2 → train with LR 3e-4
head    → train with LR 1e-3
```

We create a new optimizer after unfreezing block 2.

In [ ]:
# WHAT: Unfreeze block 2 and create two learning-rate groups.
# WHY: The pretrained block should change more gently than the new head.
# OUTPUT: Trainable count and optimizer group learning rates.

partial_model = deepcopy(frozen_model)
for parameter in partial_model.block1.parameters():
    parameter.requires_grad = False
for parameter in partial_model.block2.parameters():
    parameter.requires_grad = True
for parameter in partial_model.head.parameters():
    parameter.requires_grad = True

partial_optimizer = torch.optim.AdamW(
    [
        {"params": partial_model.block2.parameters(), "lr": 3e-4},
        {"params": partial_model.head.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4,
)

print("total/trainable:", parameter_counts(partial_model))
print("learning rates:", [group["lr"] for group in partial_optimizer.param_groups])

In [ ]:
# WHAT: Fine-tune the last backbone block and target head.
# WHY: Partial fine-tuning adapts late features while preserving early features.
# OUTPUT: Partial-fine-tuning validation loss and accuracy.

partial_result = fit(
    partial_model,
    target_train_loader,
    target_valid_loader,
    partial_optimizer,
    epochs=22,
    use_augmentation=True,
    frozen_modules=(partial_model.block1,),
)
print("partial fine-tuning:", partial_result | {"history": "omitted"})

## 14. Compare and select using validation

Validation chooses the final strategy. Transfer learning is useful only if the evidence supports it.

In [ ]:
# WHAT: Compare the three target runs and select the lowest validation loss.
# WHY: Test data must not influence the strategy choice.
# OUTPUT: Comparison table, validation curves, and selected model name.

experiments = {
    "scratch": (scratch_model, scratch_result),
    "frozen_transfer": (frozen_model, frozen_result),
    "partial_fine_tuning": (partial_model, partial_result),
}

print(f"{'run':22s} {'val loss':>9s} {'val acc':>8s} {'trainable':>10s} {'seconds':>8s}")
print("-" * 62)
for name, (candidate, result) in experiments.items():
    _, trainable = parameter_counts(candidate)
    print(
        f"{name:22s} {result['valid_loss']:9.4f} {result['valid_accuracy']:8.3f} "
        f"{trainable:10d} {result['seconds']:8.2f}"
    )

selected_name = min(experiments, key=lambda name: experiments[name][1]["valid_loss"])
selected_model = experiments[selected_name][0]
print("\nselected from validation:", selected_name)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, (_, result) in experiments.items():
    epochs_run = range(1, len(result["history"]["valid_loss"]) + 1)
    axes[0].plot(epochs_run, result["history"]["valid_loss"], label=name)
    axes[1].plot(epochs_run, result["history"]["valid_accuracy"], label=name)
axes[0].set(title="Validation loss", xlabel="epoch", ylabel="loss")
axes[1].set(title="Validation accuracy", xlabel="epoch", ylabel="accuracy")
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
plt.tight_layout()

## 15. Final test evaluation

This is the first use of target test data. We report overall accuracy, per-route metrics, and a confusion matrix.

In [ ]:
# WHAT: Evaluate the validation-selected model once on target test data.
# WHY: Final test results should come after all model choices are complete.
# OUTPUT: Test metrics and a labeled confusion matrix.

test_result = evaluate(selected_model, target_test_loader)
print("selected model:", selected_name)
print(f"test loss={test_result['loss']:.4f}, accuracy={test_result['accuracy']:.3f}\n")
print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        target_names=ROUTE_NAMES,
        digits=3,
        zero_division=0,
    )
)

matrix = confusion_matrix(test_result["labels"], test_result["predictions"])
fig, axis = plt.subplots(figsize=(5, 4))
image = axis.imshow(matrix, cmap="Blues")
axis.set_xticks(range(3), ["A", "B", "C"])
axis.set_yticks(range(3), ["A", "B", "C"])
axis.set_xlabel("predicted route")
axis.set_ylabel("actual route")
axis.set_title("Target test confusion matrix")
for row in range(3):
    for column in range(3):
        axis.text(column, row, matrix[row, column], ha="center", va="center")
fig.colorbar(image, ax=axis)
plt.tight_layout()

## 16. Save and reload

Save the weights together with the preprocessing and class meaning. Then rebuild the model and confirm that the logits match.

In [ ]:
# WHAT: Save model state plus preprocessing metadata and verify a reload.
# WHY: Weights alone do not describe the complete inference contract.
# OUTPUT: Checkpoint path and maximum difference between original/reloaded logits.

checkpoint_path = Path(tempfile.gettempdir()) / "parcel_route_model.pt"
torch.save(
    {
        "model_state": selected_model.state_dict(),
        "architecture": {"name": "SmallTransferCNN", "num_classes": 3},
        "class_names": ROUTE_NAMES,
        "input_shape": [1, 8, 8],
        "input_range": [0.0, 1.0],
        "input_mean": SOURCE_MEAN,
        "input_std": SOURCE_STD,
        "selected_run": selected_name,
    },
    checkpoint_path,
)

loaded = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
reloaded_model = SmallTransferCNN(loaded["architecture"]["num_classes"]).to(DEVICE)
reloaded_model.load_state_dict(loaded["model_state"])
reloaded_model.eval()

raw_examples = next(iter(target_test_loader))[0][:4]
normalized_examples = normalize_batch(raw_examples).to(DEVICE)
with torch.inference_mode():
    original_logits = selected_model.eval()(normalized_examples)
    reloaded_logits = reloaded_model(normalized_examples)

max_difference = float((original_logits - reloaded_logits).abs().max())
print("checkpoint:", checkpoint_path)
print("maximum logit difference:", max_difference)
assert torch.allclose(original_logits, reloaded_logits, atol=1e-7)

## 17. Common mistakes

| Mistake | Correction |
|---|---|
| flip digit images | use label-safe transforms |
| random validation augmentation | keep evaluation fixed |
| confuse input normalization and BatchNorm | before model versus inside model |
| forget `model.eval()` | use it for validation and inference |
| freeze wrong parameters | print trainable names/count |
| unfreeze but keep old optimizer | create a new optimizer |
| use large LR for backbone | use a smaller LR than the head |
| softmax before `CrossEntropyLoss` | send raw logits |

## 18. Practice

1. Remove target augmentation and compare validation accuracy.
2. Make translation too strong, visualize it, and explain why it is unsafe.
3. Compare BatchNorm output in `train()` and `eval()` modes.
4. Change the backbone LR from `3e-4` to `1e-3`.
5. Unfreeze the complete backbone and compare validation loss.
6. Replace the tiny backbone with pretrained ResNet-18 and use `weights.transforms()`.

## 19. Recap

```text
AUGMENTATION
realistic label-safe changes during training

INPUT NORMALIZATION
prepare image values before the model

BATCHNORM
control hidden values
train mode uses batch; eval mode uses saved statistics

TRANSFER LEARNING
reuse backbone → replace head → train head → fine-tune gently
```

**Remember:** Start simple, inspect examples, and let validation decide whether transfer or fine-tuning helps.